# Day 15: Self-Attention — The Heart of Transformers

**Goal:** Build self-attention from scratch and understand the single most important mechanism in modern AI.

### The big idea

For each token in a sentence, look at ALL the other tokens and decide:
1. How much does each one matter to me right now?
2. Pull in their info, weighted by importance

```
"The cat that I saw yesterday sat on the mat"
                                ↑
                  When processing "sat":
                    look at "cat"        — HIGH (subject)
                    look at "I saw"      — LOW (irrelevant clause)
                    look at "yesterday"  — LOW (time word)
                  Combine: weighted average of all words
```

### Plan for today

1. The intuition with food analogies
2. Compute attention by hand on tiny vectors (so you SEE every number)
3. Build it in code with Q, K, V matrices
4. Add causal masking (no peeking at future tokens)
5. Visualize attention patterns
6. Compare to averaging (Day 11) and recurrence (Day 13)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. The Query / Key / Value Trick — A Food Analogy

Imagine you're at a buffet, picking what to eat. You have a craving (query). The food labels say what they are (keys). The actual food is the value.

```
Your CRAVING (query):        "spicy + savory"

Food labels (keys):           Actual food (values):
  "pad thai"  (spicy, savory) → noodles
  "cake"      (sweet)          → cake
  "salad"     (savory, fresh)  → salad

Match craving to each label → get attention scores:
  pad thai  → HIGH match
  cake      → LOW match
  salad     → MEDIUM match

Put proportional amounts on your plate:
  Result = 0.6 × noodles + 0.05 × cake + 0.35 × salad
```

That's attention. Each token = a hungry customer at the buffet, deciding what to take from the other tokens.

In code:

```
Q ("craving")  ─┐
                ├── score = Q · K → softmax → weights
K ("label")    ─┘                                  │
                                                    │
V ("food")     ─────────── weighted average ←──────┘
```

## 2. Worked Example — By Hand With Tiny Vectors

3 tokens, 2-dim each. For maximum clarity, we'll skip the Q/K/V projections and use the raw embeddings as all three.

In [ ]:
# Step 1: tiny example with 3 tokens of dim 2

X = torch.tensor([
    [1.0, 0.0],     # token 0
    [0.0, 1.0],     # token 1
    [1.0, 1.0],     # token 2
])
print(f"Embeddings X (3 tokens × 2 dims):\n{X}\n")

# For demo, Q = K = V = X (skip projections — we'll add them next)
Q, K, V = X, X, X

# Step 2: compute attention scores = Q @ K.T
scores = Q @ K.T
print(f"Scores = Q @ K.T  (3 × 3):")
print(f"  scores[i, j] = how much token i 'matches' token j")
print(scores)

# Step 3: scale by sqrt(d_k) — this is the "scaled dot product" trick
d_k = Q.size(-1)
scaled_scores = scores / (d_k ** 0.5)
print(f"\nScaled scores (divide by sqrt({d_k}) = {d_k**0.5:.3f}):")
print(scaled_scores)

# Step 4: softmax each row → weights summing to 1
weights = F.softmax(scaled_scores, dim=-1)
print(f"\nWeights after softmax (each ROW sums to 1):")
print(weights)
print(f"\nRow sums (sanity check): {weights.sum(dim=-1).tolist()}")

# Step 5: weighted sum of values
output = weights @ V
print(f"\nOutput = weights @ V:")
print(output)

print("\n--- What this means ---")
print(f"Token 0 attends [{weights[0,0]:.2f}, {weights[0,1]:.2f}, {weights[0,2]:.2f}]")
print(f"  → 0.42 of itself + 0.16 of token 1 + 0.42 of token 2")
print(f"Token 2 attends [{weights[2,0]:.2f}, {weights[2,1]:.2f}, {weights[2,2]:.2f}]")
print(f"  → most attention to itself (its Q matched its K best)")

### What just happened

You computed self-attention completely by hand:

```
1. Score:    every token's query · every token's key
2. Scale:    divide by sqrt(dim) to keep numbers tame
3. Softmax:  turn scores into probability-like weights
4. Combine:  weighted average of all the values
```

That's it. Five lines of math = the heart of every transformer.

---

## 3. Adding Q, K, V Projections — The Real Deal

In real attention, Q/K/V aren't the raw embeddings — they're LEARNED projections. Why?

- Lets the model project info into different "subspaces" for different purposes
- Q-space holds "what I'm asking"
- K-space holds "what I match against"
- V-space holds "what I actually carry"

Each is a separate `nn.Linear` (no bias, by convention).

In [ ]:
# Real single-head self-attention (without causal mask yet)

class SelfAttention(nn.Module):
    def __init__(self, embed_dim, head_dim):
        super().__init__()
        # Three separate projections — each transforms (embed_dim → head_dim)
        self.W_q = nn.Linear(embed_dim, head_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, head_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, head_dim, bias=False)
        self.head_dim = head_dim
    
    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        Q = self.W_q(x)                                    # (B, T, head_dim)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Compute scores: (B, T, head_dim) @ (B, head_dim, T) → (B, T, T)
        scores = Q @ K.transpose(-2, -1)
        scores = scores / (self.head_dim ** 0.5)            # scale
        
        # Softmax → weights
        weights = F.softmax(scores, dim=-1)                  # (B, T, T)
        
        # Weighted sum of values
        out = weights @ V                                    # (B, T, head_dim)
        return out, weights

# Try it with one tiny "sentence" of 4 tokens with 8-dim embeddings
torch.manual_seed(42)
attn = SelfAttention(embed_dim=8, head_dim=8)

batch_size = 1
seq_len = 4
x = torch.randn(batch_size, seq_len, 8)

out, weights = attn(x)
print(f"Input shape:   {x.shape}    (batch, seq_len, embed_dim)")
print(f"Output shape:  {out.shape}    (batch, seq_len, head_dim)")
print(f"Weights shape: {weights.shape}    (batch, seq_len, seq_len)")
print(f"\nAttention weights (each row sums to 1):")
print(weights[0])
print(f"\nRow sums: {weights[0].sum(dim=-1).tolist()}")

## 4. Causal Masking — No Peeking at the Future!

When predicting the next token, position `t` should ONLY see positions `0, 1, ..., t`. Looking at future tokens would be cheating (the model would see the answer during training).

We enforce this by **setting future scores to -infinity** before softmax. After softmax, `e^(-inf) = 0`, so future positions get 0 weight.

The mask is just a lower-triangular matrix:

```
mask:
[[1, 0, 0, 0],
 [1, 1, 0, 0],
 [1, 1, 1, 0],
 [1, 1, 1, 1]]

Where 1 = visible, 0 = blocked.
Row i = "positions that token i can attend to".
```

In [ ]:
# Add a causal mask to our attention module

class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, head_dim, max_seq_len=64):
        super().__init__()
        self.W_q = nn.Linear(embed_dim, head_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, head_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, head_dim, bias=False)
        self.head_dim = head_dim
        
        # Precompute the causal mask (lower-triangular ones)
        # register_buffer = "save with the model but don't train"
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(max_seq_len, max_seq_len))
        )
    
    def forward(self, x):
        B, T, _ = x.shape
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        scores = Q @ K.transpose(-2, -1)
        scores = scores / (self.head_dim ** 0.5)
        
        # Apply causal mask: set FUTURE positions to -inf
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        
        weights = F.softmax(scores, dim=-1)
        out = weights @ V
        return out, weights

# Test causal attention
torch.manual_seed(42)
causal_attn = CausalSelfAttention(embed_dim=8, head_dim=8)

x = torch.randn(1, 4, 8)
out, weights = causal_attn(x)

print(f"Causal attention weights:\n{weights[0]}")
print(f"\nNotice: UPPER TRIANGLE is 0!")
print(f"  - Row 0 (token 0): can only attend to itself")
print(f"  - Row 1 (token 1): attends to tokens 0, 1")
print(f"  - Row 2 (token 2): attends to tokens 0, 1, 2")
print(f"  - Row 3 (token 3): attends to all 4 tokens")

## 5. Visualize Attention Patterns

Pictures help A LOT for attention. Let's plot the attention matrix for a longer sequence:

In [ ]:
# Visualize causal attention weights for a 10-token sequence

torch.manual_seed(0)
attn_full = SelfAttention(embed_dim=16, head_dim=16)
attn_causal = CausalSelfAttention(embed_dim=16, head_dim=16, max_seq_len=10)

# Copy weights so both modules behave identically
attn_causal.W_q.weight.data = attn_full.W_q.weight.data.clone()
attn_causal.W_k.weight.data = attn_full.W_k.weight.data.clone()
attn_causal.W_v.weight.data = attn_full.W_v.weight.data.clone()

x = torch.randn(1, 10, 16)
_, weights_full = attn_full(x)
_, weights_causal = attn_causal(x)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(weights_full[0].detach().numpy(), cmap='Blues')
axes[0].set_title('Full self-attention\n(token can look anywhere)')
axes[0].set_xlabel('Key position (attending TO)')
axes[0].set_ylabel('Query position (attending FROM)')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(weights_causal[0].detach().numpy(), cmap='Blues')
axes[1].set_title('Causal self-attention\n(can only look at past + self)')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

print("Reading the heatmap:")
print("  Row i = token i's attention distribution")
print("  Bright cells = high attention")
print("\nIn the causal version, the upper triangle is BLACK = blocked.")
print("Notice early tokens have fewer options → must attend more heavily to itself.")

## 6. Why "Scaled" Dot-Product? The Numerical Stability Trick

The scaling by `1 / sqrt(d_k)` exists for a real reason. Let's see why:

In [ ]:
# Demonstrate why we scale by sqrt(d_k)

torch.manual_seed(0)

# Try different head_dim sizes and look at score magnitudes
dims_to_test = [4, 16, 64, 256]

fig, axes = plt.subplots(1, len(dims_to_test), figsize=(16, 4))

for ax, d in zip(axes, dims_to_test):
    # Random Q and K vectors with standard scale
    Q = torch.randn(10, d)
    K = torch.randn(10, d)
    
    # Unscaled scores
    unscaled = Q @ K.T
    
    # Scaled scores
    scaled = unscaled / (d ** 0.5)
    
    # Show distributions
    ax.hist(unscaled.flatten().numpy(), bins=30, alpha=0.5, label='Unscaled', color='red')
    ax.hist(scaled.flatten().numpy(), bins=30, alpha=0.5, label='Scaled by 1/√d', color='blue')
    ax.set_title(f'd = {d}')
    ax.set_xlabel('Score value')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Without scaling, score magnitudes blow up as dim grows')
plt.tight_layout()
plt.show()

print("Why scaling matters:")
print("  At d=256, unscaled scores easily reach ±30+.")
print("  After softmax, very large numbers create one HUGE probability and zeros for the rest.")
print("  That makes gradients vanish — the model can't learn.")
print("\n  Dividing by sqrt(d) keeps scores roughly the same scale regardless of dimension.")

## 7. Train Attention on Real Data — Bigram + Attention

Let's beat yesterday's bigram model by giving it attention instead of just a lookup table. Same dataset, same task — but now each token can see all previous tokens.

In [ ]:
# Same Shakespeare corpus as Day 14

text = """to be or not to be that is the question
whether tis nobler in the mind to suffer
the slings and arrows of outrageous fortune
or to take arms against a sea of troubles
and by opposing end them to die to sleep
no more and by a sleep to say we end
the heart ache and the thousand natural shocks
that flesh is heir to tis a consummation
devoutly to be wished to die to sleep
to sleep perchance to dream ay there's the rub
for in that sleep of death what dreams may come
when we have shuffled off this mortal coil
must give us pause there's the respect
that makes calamity of so long life"""

chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

print(f"Vocab size: {vocab_size}")
print(f"Corpus length: {len(data)}")

# Hyperparameters
BLOCK_SIZE = 16    # how many chars of context we use
BATCH_SIZE = 32
EMBED_DIM = 32

def get_batch():
    starts = torch.randint(0, len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[s : s + BLOCK_SIZE] for s in starts])
    y = torch.stack([data[s+1 : s+BLOCK_SIZE+1] for s in starts])
    return x, y

xb, yb = get_batch()
print(f"\nBatch: x={xb.shape}, y={yb.shape}")

In [ ]:
# A simple language model: Embedding → Causal Self-Attention → Linear head
# Compare against Day 14's bigram (just Embedding → output)

class AttentionLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, block_size=16):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.attn = CausalSelfAttention(embed_dim, embed_dim, max_seq_len=block_size)
        self.head = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, idx, targets=None):
        # idx: (B, T)
        x = self.tok_emb(idx)                       # (B, T, embed_dim)
        x, _ = self.attn(x)                         # (B, T, embed_dim)
        logits = self.head(x)                       # (B, T, vocab_size)
        
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss

# Train
torch.manual_seed(42)
model = AttentionLM(vocab_size, EMBED_DIM, BLOCK_SIZE)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

losses = []
for step in range(2000):
    xb, yb = get_batch()
    _, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 400 == 0:
        print(f"Step {step:4d}: loss={loss.item():.4f}")

print(f"\nFinal loss: {losses[-1]:.4f}")
print(f"Day 14 bigram final loss was around 2.0–2.2 for the same dataset.")
print(f"Attention can use context → lower loss = better predictions.")

In [ ]:
# Generate text — same loop as Day 14, but the model uses attention

def generate(model, start_char='t', max_new=200, temperature=1.0):
    model.eval()
    idx = torch.tensor([[char_to_idx[start_char]]], dtype=torch.long)
    out = [start_char]
    with torch.no_grad():
        for _ in range(max_new):
            # Crop to block_size (model's attention range)
            idx_crop = idx[:, -BLOCK_SIZE:]
            logits, _ = model(idx_crop)
            last_logits = logits[0, -1, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()
            out.append(idx_to_char[next_idx])
            idx = torch.cat([idx, torch.tensor([[next_idx]])], dim=1)
    return ''.join(out)

print("--- Generated text (attention-based model) ---\n")
torch.manual_seed(0)
print(generate(model, start_char='t', max_new=300))

## 8. Inspecting What the Model Learned

Pass a real sequence through and visualize the attention weights. We can SEE what each character attends to.

In [ ]:
# Visualize what the trained model attends to

sample = "to be or not to "      # 16 chars exactly = BLOCK_SIZE
sample_ids = torch.tensor([[char_to_idx[c] for c in sample]], dtype=torch.long)

model.eval()
with torch.no_grad():
    x = model.tok_emb(sample_ids)
    _, attn_weights = model.attn(x)

weights_np = attn_weights[0].numpy()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(weights_np, cmap='Blues')
plt.colorbar(im, ax=ax, label='Attention weight')

ax.set_xticks(range(len(sample)))
ax.set_yticks(range(len(sample)))
ax.set_xticklabels([repr(c) for c in sample], fontsize=10)
ax.set_yticklabels([repr(c) for c in sample], fontsize=10)
ax.set_xlabel('Key (attending TO this position)')
ax.set_ylabel('Query (attending FROM this position)')
ax.set_title(f"Trained attention weights for '{sample}'")
plt.tight_layout()
plt.show()

print("Each row shows where that token 'looks' to make its prediction.")
print("Lower-triangular pattern = causal mask (can't see future).")
print("Brighter cells = stronger attention.")
print("\nIn a tiny model on tiny data, patterns are noisy.")
print("In real LLMs, you'd see things like 'verb' attending to 'subject',")
print("'pronoun' attending to its referent, closing brackets attending to opening, etc.")

---

## Exercises

1. **Verify attention math by hand:** Take a 2-token sequence with dim 2, choose your own Q, K, V values, compute attention manually. Then run the same numbers through `SelfAttention` and verify outputs match.

2. **Effect of removing mask:** Take `CausalSelfAttention` and remove the mask. Train the LM on Shakespeare. Does the loss go to ZERO (because the model can cheat by looking ahead at the target)?

3. **Tiny attention layer:** Try `head_dim=4`. How does the loss compare? What about `head_dim=128`?

4. **Visualize before vs after training:** Run a sequence through the model BEFORE training, save the attention weights, then train and compare. Trained weights should be more structured.

5. **Different sequences, same model:** Visualize attention for "the cat sat on" vs "to be or not to". Does the model attend to different patterns?

---

## Key Takeaways

### The complete recipe

```python
Q = x @ W_q                   # what each token is looking for
K = x @ W_k                   # what each token offers
V = x @ W_v                   # what each token carries

scores = Q @ K.T / sqrt(d)    # scaled similarity matrix (T × T)
scores = mask_future(scores)  # set future to -inf (for causal LMs)
weights = softmax(scores)     # turn into probability weights
output = weights @ V          # weighted average of values
```

That's a complete attention layer. Memorize this — it's the heart of every transformer.

### Why this beats RNNs

| | RNN | Self-Attention |
|---|---|---|
| Process order | Sequential (slow) | Parallel (fast on GPU) |
| Long-range info | Vanishing gradient | Direct lookup, same cost |
| Interpretable | Hidden state opaque | Attention weights are visible |
| Foundation of | LSTMs, GRUs | GPT, BERT, Claude, Llama |

### Where we are in the journey

```
Day 13: RNN — first sequence model                       ✓
Day 14: Bigram LM — the next-token task                  ✓
Day 15: SELF-ATTENTION — look at past, weighted          ✓ ← YOU ARE HERE
Day 16: Multi-head attention — many heads in parallel
Day 17: Positional encoding — telling the model about ORDER
Day 18: Project — attention-based text generator
Day 19+: Transformer block (it all comes together)
```

**Tomorrow:** Multi-head attention. Instead of one attention computation, run several in parallel — each can learn a different "kind" of attention (syntactic, semantic, distance-based, etc.). This is what transformers actually use.